In [1]:
%load_ext autoreload
%autoreload 2
#%export SETUPTOOLS_USE_DISTUTILS=stdlib
#%cd /home/felix/Desktop/AA_Uni/Projektarbeit/test/Genesis-Dog-Walking/test

import genesis as gs
import logging
import utils
import torch
import os

#if torch.cuda.is_available():
#    gs.init(logging_level=logging.WARNING, backend=gs.gpu)
#else:
gs.init(logging_level=logging.WARNING, backend=gs.gpu)

from buffer import Buffer
from network import Network
from make_environment import Go2WalkingEnv
from reward import Rewards

path = os.getcwd()



: 

In [2]:
reward_fn = Rewards()
num_envs = 512
max_steps = 100
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

env = Go2WalkingEnv(
    num_envs=num_envs,
    device=device,
    show_viewer=False,
    use_terrain=False,  
    episode_length_s=20.0,
    reward_fn=reward_fn,
    min_up_dot=0.05
)
print(device)

[Genesis] [18:00:27] [WARNING] Viewer option 'n_rendered_envs' is deprecated and will be removed in a future release. Please use 'rendered_envs_idx' instead.


[Genesis] [18:00:35] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
cuda


In [3]:
lin_vel_x = 0.4
env.set_commands(lin_vel_x=lin_vel_x, lin_vel_y=0.0, ang_vel_yaw=0.0)
policy = Network(
    num_outputs=env.num_actions,
    num_inputs=env.num_obs,
    gamma=0.99,
    lmbda=0.0,
    epsilon=0.1,
)
buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=max_steps,
    device=device
)

In [4]:
import torch
policy.apply(lambda m: torch.nn.init.xavier_uniform_(m.weight) if hasattr(m, 'weight') else None)

Network(
  (shared): Sequential(
    (0): Linear(in_features=48, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
  )
  (actor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (actor_mean): Linear(in_features=32, out_features=12, bias=True)
  (critic): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [5]:

def adjust_motion_command(total_lin_reward, lin_reward_t, i, lin_vel_x, optim=None, policy=None):
    counter = i % 20 # should restart and replace list every 20 update iterations
    total_lin_reward[counter] = lin_reward_t.unsqueeze(-1) # add batch dimension to lin_reward_t and store in total_lin_reward list at index counter
    # get mean from lin_vel_reward and adjust the command accordingly
    mean_lin_vel_reward = total_lin_reward.mean().item()
    # if devieates
    if mean_lin_vel_reward >= 0.85: 
        lin_vel_x += 0.05
        env.set_commands(lin_vel_x=lin_vel_x, lin_vel_y=0.0, ang_vel_yaw=0.0)
        print(utils.save_checkpoint(
            path=f"{path}/checkpoints/Safe_Before_Change_go2_update_{i}.pt",
            policy=policy,
            optim=optim,
            update=i,
            avg_rew=buffer.rewards.mean().item()))
    return total_lin_reward, lin_vel_x

        
        

    

In [6]:
def adjust_scales(update):
    if update < 15:
        scales = { # No walking reward, High termiantion penaltym, focus on not falling and not moving sideways
            "tracking_lin_vel_x": 0.0,
            "tracking_ang_vel": 0.2,
            "x_progress": 0.0,
            "lin_vel_z": -0.1,
            "lin_vel_y": -0.05,
            "action_rate": -0.002,
            "similar_to_default": -1.0,
            "termination": -6.0,
            "sideway_movement": -0.2,
            "base_height": 0.1,
        }
    elif update < 50:
        scales = {
            "tracking_lin_vel_x": 0.0,
            "tracking_ang_vel": 0.0,
            "x_progress": 0.0,
            "lin_vel_z": -0.2,
            "lin_vel_y": -0.1,
            "action_rate": -0.002,
            "similar_to_default": -0.5,
            "termination": -6.0,
            "sideway_movement": -0.1,
            "base_height": 0.1,
        }

    else:
        scales = {
            "tracking_lin_vel_x": 1.0,
            "tracking_ang_vel": 1.0,
            #"x_progress": 0.5,
            "lin_vel_z": -1.0,
            "lin_vel_y": -5.0,
            "action_rate": -0.005,
            "similar_to_default": -0.1,
            #"termination": -4.0,
            "sideway_movement": -1.0,
            #"base_height": 0.1,
        }

    reward_fn.scales = scales

In [7]:
scales = {
            "tracking_lin_vel_x": 1.0,
            "tracking_ang_vel": 1.0,
            "lin_vel_z": -1.0,
            "lin_vel_y": -5.0,
            "action_rate": -0.005,
            "similar_to_default": -0.1,
            "sideway_movement": -1.0,

        }

reward_fn.scales = scales

In [8]:
optim = torch.optim.Adam(policy.parameters(), lr=5e-4, eps=1e-8)
#torch.autograd.set_detect_anomaly(True)
num_updates = 1000
steps_per_update = 128
update_epochs = 50
minibatch_size = 512
start_update = 0
total_lin_reward = torch.zeros((20, steps_per_update, num_envs, 1), device=device)
lin_vel_x = 0.4

buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=steps_per_update,
    device=device
)
if start_update > 0:
    print(f"Loading checkpoint from update {start_update}")
    checkpoint = torch.load(f"{path}/checkpoints/go2_update_{start_update}.pt", map_location=device)
    policy.load_state_dict(checkpoint["model_state_dict"])
    optim.load_state_dict(checkpoint["optimizer_state_dict"])
    #lin_vel_x = checkpoint['lin_vel_x']


for i in range(start_update, num_updates):
    #adjust_scales(i)
    print(f"Running Sim: i: {i}")
    with torch.no_grad():
        buffer.reset()
        obs = env.reset()
        buffer.init_obs(obs, policy.get_value(obs))
        for step in range(steps_per_update):
            obs = obs.to(device)
            actions, value = policy.get_actions(obs)
            log_probs, log_probs_value, entropy = policy.compute_log_probs(obs, actions)
            next_obs , reward, done, info = env.step(actions)
            reward, lin_vel_reward = reward
            buffer.add_step(next_obs, actions, log_probs, reward, done, value, lin_vel_reward)

            obs = next_obs

            if done.any():
                obs = env.reset()

        buffer.compute_returns_and_advantages(gamma=0.99, lmbda=0.95)
    total_lin_reward, lin_vel_x = adjust_motion_command(total_lin_reward, lin_vel_reward, i, lin_vel_x, optim=optim, policy=policy)
    print(f"Running Epochs: i: {i}, Avg_reward: {buffer.rewards.mean():.3f}")
    for epoch in range(update_epochs):
        batch = buffer.get_batch(minibatch_size)

        log_probs_new, values_new, entropy = policy.compute_log_probs(batch['obs'], batch['actions'])
        #print(batch)
        critic_loss, actor_loss = policy.compute_loss(states= batch['obs'],
                                                      actions= batch['actions'],
                                                      advantages=batch['advantages'],
                                                      critic_targets=batch['values'],
                                                      log_probs_old=batch['log_probs'],
                                                      returns = batch['returns'],)
        entropy_loss = -0.01 * entropy.mean()

        loss = actor_loss + 0.5 * critic_loss + entropy_loss

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5) # gradient clipping
        optim.step()

    if i % 10 == 0:
        print(utils.save_checkpoint(
            path=f"{path}/checkpoints/go2_update_{i}.pt",
            policy=policy,
            optim=optim,
            update=i,
            avg_rew=buffer.rewards.mean().item()))
        print(utils.make_eval_video(
            env=env,
            policy=policy,
            filename=f"{path}/video/eval_update_{i}.mp4",
            eval_steps=600,
        ))


Running Sim: i: 0
Running Epochs: i: 0, Avg_reward: 1.855
None
{'video_path': '/home/felix/uni/Projektarbeit/test/Genesis-Dog-Walking/test/video/eval_update_0.mp4', 'episode_reward': 1702.5222440958023, 'steps': 600}
Running Sim: i: 1
Running Epochs: i: 1, Avg_reward: 3.043
Running Sim: i: 2
Running Epochs: i: 2, Avg_reward: 2.286
Running Sim: i: 3
Running Epochs: i: 3, Avg_reward: 2.853
Running Sim: i: 4
Running Epochs: i: 4, Avg_reward: 2.853
Running Sim: i: 5
Running Epochs: i: 5, Avg_reward: 2.853
Running Sim: i: 6
Running Epochs: i: 6, Avg_reward: 2.853
Running Sim: i: 7
Running Epochs: i: 7, Avg_reward: 2.864
Running Sim: i: 8
Running Epochs: i: 8, Avg_reward: 1.864
Running Sim: i: 9
Running Epochs: i: 9, Avg_reward: 2.573
Running Sim: i: 10
Running Epochs: i: 10, Avg_reward: 2.573
None
{'video_path': '/home/felix/uni/Projektarbeit/test/Genesis-Dog-Walking/test/video/eval_update_10.mp4', 'episode_reward': 1640.553300857544, 'steps': 600}
Running Sim: i: 11
Running Epochs: i: 11, 

In [9]:
utils.save_checkpoint(
        path=f"/Users/felix/PycharmProjects/Genesis-Dog-Walking/test/go2_update_Final.pt",
        policy=policy,
        optim=optim,
        update=100,
        avg_rew=buffer.rewards.mean().item(),
    )


PermissionError: [Errno 13] Permission denied: '/Users'